## **PersonaPath Phase 2 Web App**


In [1]:
!pip uninstall -y whisper
!pip install -q flask librosa opencv-python-headless moviepy openai-whisper huggingface_hub tensorflow numpy werkzeug cmudict python-Levenshtein
!apt-get update -qq && apt-get install -y -qq ffmpeg


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
import os

os.makedirs("templates", exist_ok=True)

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>PersonaPath: Multi-Modal Speech Assessment</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=Lexend:wght@600;700;800&display=swap" rel="stylesheet">
    <style>
        body { font-family: 'Inter', sans-serif; background-color: #f1f5f9; }
        .font-display { font-family: 'Lexend', sans-serif; }
        .coaching-bubble-tail::after {
            content: '';
            position: absolute;
            bottom: -10px;
            left: 30px;
            width: 20px;
            height: 20px;
            background: white;
            clip-path: polygon(0 0, 100% 0, 50% 100%);
        }
    </style>
</head>
<body class="h-screen w-screen flex flex-col overflow-hidden">

  <!-- Navbar -->
  <header class="h-[64px] bg-white border-b border-slate-200 flex items-center justify-between px-6 shrink-0 z-10">
     <div class="text-2xl font-black text-[#00288e] font-display tracking-tight">PersonaPath</div>
     <div class="flex gap-6 font-bold text-sm h-full items-end">
        <button id="tab-phase1" class="text-slate-400 pb-4 border-b-2 border-transparent transition cursor-pointer">Phase 1: Literacy Coach</button>
        <button id="tab-phase2" class="text-[#00288e] border-b-2 border-[#00288e] pb-4 transition cursor-pointer">Phase 2: Presentation Pro</button>
     </div>
     <div class="flex gap-4 items-center">
        <div class="bg-teal-50 border border-teal-100 text-teal-700 px-3 py-1 rounded-full text-[10px] font-bold flex items-center gap-1.5 shadow-sm">
            <span class="w-1.5 h-1.5 rounded-full bg-teal-500 animate-pulse"></span> ACTIVE SESSION
        </div>
        <div class="w-8 h-8 bg-slate-200 rounded-full"></div>
     </div>
  </header>

  <!-- Container -->
  <div class="flex flex-grow min-h-0">
     
     <!-- MAIN DASHBOARD CONTENT -->
     <main class="flex-1 p-8 flex flex-col min-h-0 overflow-y-auto">
        
        <!-- ==================== PHASE 1 PANEL ==================== -->
        <div id="panel-phase1" class="grid grid-cols-12 gap-6 min-h-0 hidden">
           <!-- Left Column -->
           <div class="col-span-7 flex flex-col gap-6 min-h-0">
              <!-- Passage Selection and Preview -->
              <div class="bg-white rounded-xl border border-slate-200 p-6 shadow-sm flex flex-col gap-4">
                 <div class="flex justify-between items-center pb-2 border-b border-slate-100">
                    <h2 class="text-lg font-bold text-slate-800 font-display">Reading Passage</h2>
                    <span class="text-[10px] bg-blue-50 text-[#00288e] px-2.5 py-1 rounded-full font-bold uppercase tracking-wide">LEVEL: PRIMARY</span>
                 </div>
                 
                 <div class="grid grid-cols-2 gap-4">
                    <div>
                       <label class="block text-xs font-bold text-slate-500 uppercase mb-1">Select Passage</label>
                       <select id="p1-passage-select" class="w-full bg-slate-50 border border-slate-200 rounded-lg p-2 text-sm text-slate-700 font-medium">
                          <option value="Primary - The Elephant (Beginner)">Primary - The Elephant (Beginner)</option>
                          <option value="Primary - Sunny Day (Easy)">Primary - Sunny Day (Easy)</option>
                          <option value="Intermediate - Breeze (Medium)">Intermediate - Breeze (Medium)</option>
                          <option value="Advanced - Full Story (Hard)">Advanced - Full Story (Hard)</option>
                       </select>
                    </div>
                    <div class="flex items-end pb-1.5">
                       <label class="flex items-center gap-2 cursor-pointer">
                          <input type="checkbox" id="p1-custom-checkbox" class="rounded border-slate-300 text-[#00288e] focus:ring-[#00288e]">
                          <span class="text-xs font-bold text-slate-600">Use custom passage</span>
                       </label>
                    </div>
                 </div>

                 <!-- Custom Text Area (hidden by default) -->
                 <div id="p1-custom-container" class="hidden">
                    <label class="block text-xs font-bold text-slate-500 uppercase mb-1">Custom Passage Text</label>
                    <textarea id="p1-custom-text" placeholder="Type your custom passage here..." class="w-full bg-slate-50 border border-slate-200 rounded-lg p-3 text-sm text-slate-700 h-24 focus:outline-none focus:ring-1 focus:ring-blue-500"></textarea>
                 </div>

                 <!-- Reading Preview Box -->
                 <div class="p-5 bg-slate-50 rounded-xl border border-slate-100">
                    <p id="p1-passage-preview" class="text-xl font-display leading-relaxed text-slate-700 transition-all duration-300">
                       The elephant sat quietly by the river bank.
                    </p>
                 </div>
                 <div class="text-[10px] text-slate-400 font-bold uppercase tracking-wider flex items-center gap-1.5">
                    <span>⚡ TRANSFORMER-BASED ALIGNMENT (WAV2VEC 2.0)</span>
                 </div>
              </div>

              <!-- Audio Recorder / Upload -->
              <div class="bg-white rounded-xl border border-slate-200 p-6 shadow-sm flex flex-col gap-4">
                 <h3 class="text-sm font-bold text-slate-800">Record or Upload Audio</h3>
                 <div class="flex flex-col items-center justify-center border-2 border-dashed border-slate-200 rounded-xl py-8 bg-slate-50 relative overflow-hidden">
                    <!-- Standard Mic record UI -->
                    <div id="p1-audio-ui" class="flex flex-col items-center justify-center">
                       <button id="btn-p1-record" class="bg-red-50 text-red-600 w-14 h-14 rounded-full flex items-center justify-center hover:bg-red-100 transition shadow-sm border border-red-100 mb-2" title="Record Microphone">
                          <svg class="w-6 h-6" fill="currentColor" viewBox="0 0 24 24"><path d="M12 14c1.66 0 3-1.34 3-3V5c0-1.66-1.34-3-3-3S9 3.34 9 5v6c0 1.66 1.34 3 3 3zm5.3-3c0 3-2.54 5.1-5.3 5.1S6.7 14 6.7 11H5c0 3.41 2.72 6.23 6 6.72V21h2v-3.28c3.28-.48 6-3.3 6-6.72h-1.7z"/></svg>
                       </button>
                       <p class="text-xs font-bold text-slate-600 mb-1">Click microphone to record speech</p>
                       <p class="text-[10px] text-slate-400">or upload audio file below</p>
                    </div>

                    <!-- Recording active UI -->
                    <div id="p1-recording-ui" class="hidden flex flex-col items-center justify-center">
                       <button id="btn-p1-stop" class="bg-red-600 text-white px-6 py-2.5 rounded-full font-bold shadow-md hover:scale-105 transition flex items-center gap-2 mb-2">
                          <div class="w-2.5 h-2.5 bg-white rounded-sm animate-ping"></div> Stop & Analyze
                       </button>
                       <p class="text-xs font-bold text-red-600 animate-pulse">Recording Audio...</p>
                    </div>
                 </div>

                 <!-- File upload controls -->
                 <div class="flex justify-between items-center gap-4">
                    <audio id="p1-audio-player" class="hidden flex-1" controls></audio>
                    <div class="flex items-center gap-2">
                       <button id="btn-p1-upload" class="bg-slate-100 text-slate-700 px-4 py-2 rounded-lg text-xs font-bold hover:bg-slate-200 transition border border-slate-200">
                          Upload File
                       </button>
                       <input type="file" id="p1-file-upload" accept="audio/*" class="hidden">
                    </div>
                 </div>
              </div>
           </div>

           <!-- Right Column -->
           <div class="col-span-5 flex flex-col gap-6 min-h-0">
              <!-- Progress bar -->
              <div class="bg-white rounded-xl p-5 shadow-sm border border-slate-200">
                 <div class="flex justify-between text-xs font-bold text-slate-700 mb-2.5">
                    <span id="p1-loading-step">Analysis Progress</span>
                    <span id="p1-loading-pct" class="text-[#00288e]">Waiting...</span>
                 </div>
                 <div class="w-full h-2 bg-slate-100 rounded-full overflow-hidden">
                    <div id="p1-loading-bar" class="h-full bg-[#00288e] transition-all duration-300 w-0"></div>
                 </div>
              </div>

              <!-- Metrics Card -->
              <div class="bg-white rounded-xl p-5 shadow-sm border border-slate-200">
                 <div class="flex gap-6">
                    <!-- Accuracy Circle -->
                    <div class="flex-1 border-r border-slate-100 pr-6 flex flex-col">
                       <h3 class="w-full text-left text-xs font-bold text-slate-800 mb-4 pb-2 border-b border-slate-100">Pronunciation Accuracy</h3>
                       <div class="flex flex-col items-center justify-center flex-1">
                          <div class="relative w-24 h-24 mb-4">
                             <svg class="w-full h-full -rotate-90" viewBox="0 0 100 100">
                                <circle cx="50" cy="50" r="40" fill="transparent" stroke="#eff4ff" stroke-width="12"></circle>
                                <circle id="p1-acc-ring" cx="50" cy="50" r="40" fill="transparent" stroke="#006c49" stroke-width="12" stroke-dasharray="251.2" stroke-dashoffset="251.2" stroke-linecap="round" class="transition-all duration-1000 ease-out"></circle>
                             </svg>
                             <div class="absolute inset-0 flex flex-col items-center justify-center">
                                <span class="text-[1.2rem] font-black text-slate-800 tracking-tighter"><span id="p1-acc-score">0</span>%</span>
                             </div>
                          </div>
                          <div class="bg-slate-50 text-slate-500 text-[8px] font-bold px-2 py-1 rounded uppercase tracking-wide border border-slate-100 w-full text-center">Accuracy Score</div>
                       </div>
                    </div>

                    <!-- Numeric metrics -->
                    <div class="flex-1 flex flex-col">
                       <h3 class="w-full text-left text-xs font-bold text-slate-800 mb-4 pb-2 border-b border-slate-100">Details</h3>
                       <div class="flex flex-col gap-2 flex-1 justify-center">
                          <div class="bg-[#eff4ff] p-2 rounded-lg text-center flex justify-between items-center px-4">
                             <span class="text-[10px] font-bold text-slate-500 uppercase">Words/Min</span>
                             <div id="p1-wpm" class="text-lg font-black text-[#00288e]">0</div>
                          </div>
                          <div class="bg-[#eff4ff] p-2 rounded-lg text-center flex justify-between items-center px-4">
                             <span class="text-[10px] font-bold text-slate-500 uppercase">Hesitations</span>
                             <div id="p1-hesitations" class="text-lg font-black text-[#00288e]">0</div>
                          </div>
                          <div class="bg-slate-50 border border-slate-100 p-2 rounded-lg text-center flex justify-between items-center px-4">
                             <span class="text-[10px] font-bold text-slate-500 uppercase">Duration</span>
                             <div class="text-xs font-bold text-slate-700"><span id="p1-duration">0.0</span>s</div>
                          </div>
                       </div>
                    </div>
                 </div>
              </div>

              <!-- Claude's Coaching Corner -->
              <div class="bg-[#0b2885] rounded-xl p-5 text-white shadow-md flex-1 flex flex-col min-h-0">
                 <div class="flex items-center gap-3 mb-4 shrink-0">
                    <div class="w-10 h-10 rounded-full overflow-hidden border border-white/20 bg-white/10 flex items-center justify-center">
                       <svg class="w-6 h-6 text-blue-200" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M9.663 17h4.673M12 3v1m6.364 1.636l-.707.707M21 12h-1M4 12H3m3.343-5.657l-.707-.707m2.828 9.9a5 5 0 117.072 0l-.548.547A3.374 3.374 0 0014 18.469V19a2 2 0 11-4 0v-.531c0-.895-.356-1.754-.988-2.386l-.548-.547z"/></svg>
                    </div>
                    <div>
                       <h3 class="text-xs font-bold">Claude's Coaching Corner</h3>
                       <span class="text-[8px] uppercase tracking-wider text-blue-300">Active Feedback</span>
                    </div>
                 </div>
                 <div class="flex-grow flex flex-col gap-3 overflow-y-auto pr-1">
                    <div id="p1-coaching-box" class="bg-white/10 rounded-lg p-3">
                       <p class="text-[11px] text-blue-200 font-medium">Record reading to receive tutoring tips...</p>
                    </div>
                    <div class="border-t border-white/10 pt-2 shrink-0">
                       <span class="text-[8px] font-bold text-blue-300 uppercase tracking-widest block mb-1">What We Heard</span>
                       <p id="p1-transcription" class="text-[10px] text-blue-50 italic font-medium leading-relaxed">"Waiting for transcription..."</p>
                    </div>
                 </div>
              </div>
           </div>
        </div>

        <!-- ==================== PHASE 2 PANEL ==================== -->
        <div id="panel-phase2" class="grid grid-cols-12 gap-6 min-h-0">
           <!-- Left Column -->
           <div class="col-span-7 flex flex-col gap-6 min-h-0">
              <!-- Video / Webcam Container -->
              <div class="bg-[#111827] border border-slate-200 rounded-xl relative flex flex-col overflow-hidden shadow-sm flex-1 min-h-0 h-[380px]">
                  <video id="media-preview" class="w-full h-full object-contain hidden bg-[#111827]" playsinline></video>

                  <div id="video-ui" class="absolute inset-0 flex flex-col items-center justify-center bg-[#111827] z-10">
                      <div class="w-16 h-16 bg-blue-600 rounded-full flex items-center justify-center mb-4 opacity-50"><svg class="w-8 h-8 text-white" fill="currentColor" viewBox="0 0 24 24"><path d="M8 5v14l11-7z"/></svg></div>

                      <div class="absolute bottom-8 flex gap-4">
                          <button id="btn-upload" class="bg-slate-100 text-slate-700 w-12 h-12 rounded-full flex items-center justify-center hover:bg-slate-200 transition shadow-sm border border-slate-200" title="Upload Video">
                              <svg class="w-5 h-5" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M4 16v1a3 3 0 003 3h10a3 3 0 003-3v-1m-4-8l-4-4m0 0L8 8m4-4v12"></path></svg>
                          </button>
                          <input type="file" id="file-upload" accept="video/*" class="hidden">
                          <button id="btn-record" class="bg-red-50 text-red-600 w-12 h-12 rounded-full flex items-center justify-center hover:bg-red-100 transition shadow-sm border border-red-100" title="Record Camera">
                              <svg class="w-5 h-5" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M15 10l4.553-2.276A1 1 0 0121 8.618v6.764a1 1 0 01-1.447.894L15 14M5 18h8a2 2 0 002-2V8a2 2 0 00-2-2H5a2 2 0 00-2 2v8a2 2 0 002 2z"></path></svg>
                          </button>
                      </div>
                  </div>

                  <div id="recording-ui" class="absolute bottom-6 left-1/2 -translate-x-1/2 hidden z-20">
                      <button id="btn-stop" class="bg-red-600 text-white px-8 py-3 rounded-full font-bold shadow-2xl hover:scale-105 transition flex items-center gap-2">
                          <div class="w-3.5 h-3.5 bg-white rounded-sm"></div> Stop & Analyze
                      </button>
                  </div>
              </div>

              <!-- Vocal Energy Dynamics -->
              <div class="bg-white rounded-xl p-5 shrink-0 shadow-sm border border-slate-200 h-[200px] flex flex-col">
                 <div class="flex justify-between items-center mb-4">
                    <h3 class="text-sm font-bold text-slate-800">Vocal Energy Dynamics</h3>
                    <span class="text-[10px] bg-[#eff4ff] text-[#00288e] px-2.5 py-1 rounded font-bold uppercase tracking-wide">RNN/LSTM Sequence Analysis</span>
                 </div>
                 <div class="flex-1 flex items-end justify-between gap-1 pb-4 border-b border-slate-100 border-dashed relative px-4" id="energy-chart">
                    <div class="absolute inset-0 flex items-center justify-center text-xs font-semibold text-slate-400">Data populates after analysis</div>
                 </div>
                 <div class="flex justify-between text-[9px] font-bold text-slate-400 uppercase tracking-widest mt-3 px-4">
                    <span>Introduction</span>
                    <span>Core Argument</span>
                    <span>Conclusion</span>
                 </div>
              </div>
           </div>

           <!-- Right Column -->
           <div class="col-span-5 flex flex-col gap-6 min-h-0">
              <!-- Progress Bar -->
              <div class="bg-white rounded-xl p-5 shadow-sm border border-slate-200 shrink-0">
                  <div class="flex justify-between text-xs font-bold text-slate-700 mb-2.5">
                     <span id="loading-step">Analysis Progress</span>
                     <span id="loading-pct" class="text-[#00288e]">Waiting...</span>
                  </div>
                  <div class="w-full h-2 bg-[#eff4ff] rounded-full overflow-hidden">
                     <div id="loading-bar" class="h-full bg-[#00288e] transition-all duration-300 w-0"></div>
                  </div>
              </div>

              <!-- Key Metrics Card -->
              <div class="bg-white rounded-xl p-5 shadow-sm border border-slate-200 shrink-0">
                 <div class="flex gap-6">
                    <!-- Confidence Score -->
                    <div class="flex-1 border-r border-slate-100 pr-6 flex flex-col">
                       <h3 class="w-full text-left text-xs font-bold text-slate-800 mb-4 pb-2 border-b border-slate-100">Confidence Score</h3>
                       <div class="flex flex-col items-center justify-center flex-1">
                           <div class="relative w-24 h-24 mb-4">
                              <svg class="w-full h-full -rotate-90" viewBox="0 0 100 100">
                                  <circle cx="50" cy="50" r="40" fill="transparent" stroke="#eff4ff" stroke-width="12"></circle>
                                  <circle id="conf-ring" cx="50" cy="50" r="40" fill="transparent" stroke="#006c49" stroke-width="12" stroke-dasharray="251.2" stroke-dashoffset="251.2" stroke-linecap="round" class="transition-all duration-1000 ease-out"></circle>
                              </svg>
                              <div class="absolute inset-0 flex flex-col items-center justify-center">
                                  <span class="text-[1.2rem] font-black text-slate-800 tracking-tighter"><span id="conf-score">0</span>%</span>
                              </div>
                           </div>
                           <div class="bg-slate-50 text-slate-500 text-[8px] font-bold px-2 py-1 rounded uppercase tracking-wide border border-slate-100 w-full text-center">MobileNetV2</div>
                       </div>
                    </div>

                    <!-- Filler Words -->
                    <div class="flex-1 flex flex-col">
                       <h3 class="w-full text-left text-xs font-bold text-slate-800 mb-4 pb-2 border-b border-slate-100">Filler Words</h3>
                       <div class="flex flex-col gap-2 flex-1 justify-center">
                           <div class="bg-[#eff4ff] p-2 rounded-lg text-center flex justify-between items-center px-4"><span class="text-[10px] font-bold text-slate-500 uppercase">Um</span><div id="fw-um" class="text-lg font-black text-[#00288e]">0</div></div>
                           <div class="bg-[#eff4ff] p-2 rounded-lg text-center flex justify-between items-center px-4"><span class="text-[10px] font-bold text-slate-500 uppercase">Uh</span><div id="fw-uh" class="text-lg font-black text-[#00288e]">0</div></div>
                           <div class="bg-slate-50 border border-slate-100 p-2 rounded-lg text-center flex justify-between items-center px-4"><span class="text-[10px] font-bold text-slate-500 uppercase">Like</span><div id="fw-like" class="text-lg font-black text-[#ba1a1a]">0</div></div>
                       </div>
                    </div>
                 </div>
              </div>

              <!-- Presentation Insights -->
              <div class="bg-[#0b2885] rounded-xl p-5 text-white shadow-md flex-1 flex flex-col min-h-0">
                 <h3 class="text-xs font-bold flex items-center gap-2 mb-3">Presentation Insights</h3>
                 <div class="flex-1 flex flex-col gap-2 overflow-y-auto pr-1" id="insights-container">
                    <div class="bg-white/10 rounded-lg p-3">
                       <p class="text-[11px] text-blue-200 font-medium">Upload or record a video to get personalized AI coaching...</p>
                    </div>
                 </div>
              </div>
           </div>
        </div>

     </main>
  </div>

  <script>
    // Tab switching logic
    const tabPhase1 = document.getElementById('tab-phase1');
    const tabPhase2 = document.getElementById('tab-phase2');
    const panelPhase1 = document.getElementById('panel-phase1');
    const panelPhase2 = document.getElementById('panel-phase2');

    tabPhase1.onclick = () => {
        tabPhase1.className = "text-[#00288e] border-b-2 border-[#00288e] pb-4 font-bold transition cursor-pointer";
        tabPhase2.className = "text-slate-400 pb-4 border-b-2 border-transparent transition cursor-pointer hover:text-[#00288e]";
        panelPhase1.classList.remove('hidden');
        panelPhase2.classList.add('hidden');
    };

    tabPhase2.onclick = () => {
        tabPhase2.className = "text-[#00288e] border-b-2 border-[#00288e] pb-4 font-bold transition cursor-pointer";
        tabPhase1.className = "text-slate-400 pb-4 border-b-2 border-transparent transition cursor-pointer hover:text-[#00288e]";
        panelPhase2.classList.remove('hidden');
        panelPhase1.classList.add('hidden');
    };

    // Phase 1 Passages Dictionary
    const passages = {
        'Primary - The Elephant (Beginner)': 'The elephant sat quietly by the river bank.',
        'Primary - Sunny Day (Easy)': 'One sunny day I went to the park with my friends. We played games and laughed together.',
        'Intermediate - Breeze (Medium)': 'The cool breeze felt nice and the sky looked very beautiful. After playing we sat under a big tree and talked happily.',
        'Advanced - Full Story (Hard)': 'One sunny evening I went to the park with my friends. We played games ran around and laughed together. The cool breeze felt nice and the sky looked very beautiful. After playing we sat under a big tree and talked happily. It was a wonderful day and I felt very happy.'
    };

    const p1Select = document.getElementById('p1-passage-select');
    const p1CustomCheckbox = document.getElementById('p1-custom-checkbox');
    const p1CustomText = document.getElementById('p1-custom-text');
    const p1CustomContainer = document.getElementById('p1-custom-container');
    const p1Preview = document.getElementById('p1-passage-preview');

    function updatePassageText() {
        if (p1CustomCheckbox.checked) {
            p1Preview.innerText = p1CustomText.value || "Type some custom text to analyze.";
        } else {
            p1Preview.innerText = passages[p1Select.value];
        }
    }

    p1Select.onchange = updatePassageText;
    p1CustomText.oninput = updatePassageText;
    p1CustomCheckbox.onchange = () => {
        if (p1CustomCheckbox.checked) {
            p1CustomContainer.classList.remove('hidden');
        } else {
            p1CustomContainer.classList.add('hidden');
        }
        updatePassageText();
    };

    // Set initial text
    updatePassageText();

    // Phase 1 Audio Recording variables
    let p1AudioRecorder;
    let p1AudioChunks = [];
    let p1Stream;
    let isP1Recording = false;

    const btnP1Record = document.getElementById('btn-p1-record');
    const btnP1Stop = document.getElementById('btn-p1-stop');
    const p1AudioUI = document.getElementById('p1-audio-ui');
    const p1RecordingUI = document.getElementById('p1-recording-ui');
    const p1AudioPlayer = document.getElementById('p1-audio-player');
    const btnP1Upload = document.getElementById('btn-p1-upload');
    const p1FileUpload = document.getElementById('p1-file-upload');

    btnP1Upload.onclick = () => p1FileUpload.click();
    p1FileUpload.onchange = (e) => {
        if(e.target.files.length > 0) {
            const file = e.target.files[0];
            p1AudioPlayer.src = URL.createObjectURL(file);
            p1AudioPlayer.classList.remove('hidden');
            startPhase1Analysis(file);
        }
    };

    btnP1Record.onclick = async () => {
        try {
            p1Stream = await navigator.mediaDevices.getUserMedia({ audio: true });
            p1AudioPlayer.srcObject = p1Stream;
            p1AudioUI.classList.add('hidden');
            p1RecordingUI.classList.remove('hidden');
            p1AudioPlayer.classList.add('hidden');
            p1AudioPlayer.muted = true;
            p1AudioPlayer.play();

            p1AudioRecorder = new MediaRecorder(p1Stream);
            p1AudioChunks = [];

            p1AudioRecorder.ondataavailable = e => { if (e.data.size > 0) p1AudioChunks.push(e.data); };
            p1AudioRecorder.onstop = () => {
                const blob = new Blob(p1AudioChunks, { type: 'audio/wav' });
                p1AudioPlayer.srcObject = null;
                p1AudioPlayer.src = URL.createObjectURL(blob);
                p1AudioPlayer.muted = false;
                p1AudioPlayer.controls = true;
                p1AudioPlayer.classList.remove('hidden');
                p1Stream.getTracks().forEach(track => track.stop());
                p1RecordingUI.classList.add('hidden');
                p1AudioUI.classList.remove('hidden');

                const file = new File([blob], "recording.wav", { type: 'audio/wav' });
                startPhase1Analysis(file);
            };

            p1AudioRecorder.start();
            isP1Recording = true;
        } catch (err) {
            alert("Microphone access denied or unavailable.");
        }
    };

    btnP1Stop.onclick = () => {
        if (isP1Recording) {
            p1AudioRecorder.stop();
            isP1Recording = false;
        }
    };

    const setP1LoadState = (text, pct) => {
        document.getElementById('p1-loading-step').innerText = text;
        document.getElementById('p1-loading-pct').innerText = pct + '%';
        document.getElementById('p1-loading-bar').style.width = pct + '%';
    };

    async function startPhase1Analysis(file) {
        setP1LoadState('Transcribing audio (Wav2Vec)...', 20);
        document.getElementById('p1-acc-ring').style.strokeDashoffset = 251.2;
        document.getElementById('p1-acc-score').innerText = '0';

        const formData = new FormData();
        formData.append('audio', file);
        
        let refText = p1Preview.innerText;
        formData.append('reference_text', refText);

        let p = 20;
        const intv = setInterval(() => {
            if (p < 90) {
                p += 5;
                setP1LoadState('Running phonetic distance alignment...', p);
            }
        }, 800);

        try {
            const response = await fetch('/analyze_phase1', { method: 'POST', body: formData });
            clearInterval(intv);
            if (!response.ok) throw new Error('Server error');
            const data = await response.json();
            if (data.error) throw new Error(data.error);

            setP1LoadState('Analysis Complete', 100);

            // Ring offset
            const offset = 251.2 - (data.accuracy / 100) * 251.2;
            setTimeout(() => {
                document.getElementById('p1-acc-ring').style.strokeDashoffset = offset;
                animateValue('p1-acc-score', data.accuracy, 1200);
            }, 100);

            document.getElementById('p1-wpm').innerText = Math.round(data.wpm);
            document.getElementById('p1-hesitations').innerText = data.hesitations;
            document.getElementById('p1-duration').innerText = data.duration.toFixed(1);

            // Re-render text with scores
            let htmlWords = '';
            data.word_scores.forEach(ws => {
                let colorClass = 'text-slate-600 font-medium';
                let tooltip = '';
                if (ws.score === 'green') {
                    colorClass = 'text-[#006c49] font-bold';
                } else if (ws.score === 'grey') {
                    colorClass = 'text-amber-600 font-bold underline decoration-dotted';
                    tooltip = ` title="You said: ${ws.spoken}"`;
                } else if (ws.score === 'red') {
                    colorClass = 'text-[#ba1a1a] font-bold';
                    if (ws.spoken) {
                        tooltip = ` title="You said: ${ws.spoken}"`;
                    } else {
                        tooltip = ` title="Skipped/Not heard"`;
                    }
                }
                htmlWords += `<span class="${colorClass} cursor-pointer hover:underline mx-0.5" ${tooltip}>${ws.word}</span> `;
            });
            p1Preview.innerHTML = htmlWords;

            // Coaching corner
            const coachingBox = document.getElementById('p1-coaching-box');
            coachingBox.innerHTML = `<p class="text-[11px] text-blue-50 font-medium leading-tight">${data.tip}</p>`;

            // Transcription
            document.getElementById('p1-transcription').innerText = `"${data.transcription}"`;

        } catch (err) {
            clearInterval(intv);
            setP1LoadState('Error occurred', 0);
            alert('Analysis Error: ' + err.message);
        }
    }


    // Phase 2 Video Recording / Handling logic
    let mediaRecorder;
    let recordedChunks = [];
    let stream;
    let isRecording = false;

    const btnUpload = document.getElementById('btn-upload');
    const btnRecord = document.getElementById('btn-record');
    const fileUpload = document.getElementById('file-upload');
    const btnStop = document.getElementById('btn-stop');
    const videoPreview = document.getElementById('media-preview');
    const videoUI = document.getElementById('video-ui');
    const recordingUI = document.getElementById('recording-ui');

    btnUpload.onclick = () => fileUpload.click();

    fileUpload.onchange = (e) => {
        if(e.target.files.length > 0) {
            const file = e.target.files[0];
            videoPreview.src = URL.createObjectURL(file);
            videoPreview.classList.remove('hidden');
            videoUI.classList.add('hidden');
            videoPreview.controls = true;
            startAnalysis(file);
        }
    };

    btnRecord.onclick = async () => {
        try {
            stream = await navigator.mediaDevices.getUserMedia({ video: true, audio: true });
            videoPreview.srcObject = stream;
            videoPreview.classList.remove('hidden');
            videoUI.classList.add('hidden');
            recordingUI.classList.remove('hidden');
            videoPreview.controls = false;
            videoPreview.muted = true;
            videoPreview.play();

            mediaRecorder = new MediaRecorder(stream);
            recordedChunks = [];

            mediaRecorder.ondataavailable = e => { if (e.data.size > 0) recordedChunks.push(e.data); };
            mediaRecorder.onstop = () => {
                const blob = new Blob(recordedChunks, { type: 'video/webm' });
                videoPreview.srcObject = null;
                videoPreview.src = URL.createObjectURL(blob);
                videoPreview.muted = false;
                videoPreview.controls = true;
                stream.getTracks().forEach(track => track.stop());
                recordingUI.classList.add('hidden');

                const file = new File([blob], "recording.webm", { type: 'video/webm' });
                startAnalysis(file);
            };

            mediaRecorder.start();
            isRecording = true;
        } catch (err) {
            alert("Camera access denied or unavailable. Please ensure you are on HTTPS and granted permissions.");
        }
    };

    btnStop.onclick = () => {
        if (isRecording) {
            mediaRecorder.stop();
            isRecording = false;
        }
    };

    const setLoadState = (text, pct) => {
        document.getElementById('loading-step').innerText = text;
        document.getElementById('loading-pct').innerText = pct + '%';
        document.getElementById('loading-bar').style.width = pct + '%';
    };

    const animateValue = (id, end, duration) => {
        const obj = document.getElementById(id);
        let startTimestamp = null;
        const step = (timestamp) => {
            if (!startTimestamp) startTimestamp = timestamp;
            const progress = Math.min((timestamp - startTimestamp) / duration, 1);
            obj.innerHTML = (progress * end).toFixed(1);
            if (progress < 1) window.requestAnimationFrame(step);
            else obj.innerHTML = end.toFixed(1);
        };
        window.requestAnimationFrame(step);
    };

    function renderEnergyBars(contour) {
        const chart = document.getElementById('energy-chart');
        chart.innerHTML = '';
        if(!contour) return;

        contour.forEach((val, i) => {
            const bar = document.createElement('div');
            const h = Math.max(10, Math.min(100, val * 100));

            bar.className = 'flex-1 rounded-t-sm transition-all duration-1000 bg-[#00288e]';
            if(i === 0 || i === 19) bar.className = 'flex-1 rounded-t-sm transition-all duration-1000 bg-blue-100';

            bar.style.height = '0%';
            chart.appendChild(bar);

            setTimeout(() => {
                bar.style.height = h + '%';
            }, i * 50);
        });
    }

    async function startAnalysis(file) {
        setLoadState('Extracting audio & frames...', 15);
        document.getElementById('conf-ring').style.strokeDashoffset = 251.2;
        document.getElementById('conf-score').innerText = '0';
        document.getElementById('energy-chart').innerHTML = '<div class="absolute inset-0 flex items-center justify-center text-xs font-semibold text-[#00288e] animate-pulse">Analyzing...</div>';

        const formData = new FormData();
        formData.append('video', file);

        let p = 15;
        const intv = setInterval(() => {
            if (p < 90) {
                p += 2;
                setLoadState(p < 50 ? 'Running MobileNetV2...' : 'Analyzing vocal confidence (LSTM)...', p);
            }
        }, 1000);

        try {
            const response = await fetch('/analyze', { method: 'POST', body: formData });
            clearInterval(intv);
            if (!response.ok) throw new Error('Server error');
            const data = await response.json();
            if (data.error) throw new Error(data.error);

            setLoadState('Analysis Complete', 100);

            // Ring
            const offset = 251.2 - (data.adjusted_confidence / 100) * 251.2;
            setTimeout(() => {
                document.getElementById('conf-ring').style.strokeDashoffset = offset;
                animateValue('conf-score', data.adjusted_confidence, 1500);
            }, 100);

            document.getElementById('fw-um').innerText = data.filler_counts['um'] || 0;
            document.getElementById('fw-uh').innerText = data.filler_counts['uh'] || 0;
            document.getElementById('fw-like').innerText = data.filler_counts['like'] || 0;

            const insightsContainer = document.getElementById('insights-container');
            insightsContainer.innerHTML = '';

            insightsContainer.innerHTML += `
                <div class="bg-white/10 rounded-md px-3 py-2 flex justify-between gap-1 mb-0.5 shrink-0">
                    <div class="flex flex-col"><span class="text-[8px] uppercase font-bold text-blue-200 tracking-wide">Pace</span><span class="text-[11px] font-black text-white">${data.wpm} wpm</span></div>
                    <div class="flex flex-col"><span class="text-[8px] uppercase font-bold text-blue-200 tracking-wide">Silence</span><span class="text-[11px] font-black text-white">${(data.silence_ratio * 100).toFixed(0)}%</span></div>
                    <div class="flex flex-col"><span class="text-[8px] uppercase font-bold text-blue-200 tracking-wide">Pitch</span><span class="text-[11px] font-black text-white">${data.pitch_std.toFixed(0)} Hz</span></div>
                    <div class="flex flex-col"><span class="text-[8px] uppercase font-bold text-blue-200 tracking-wide">Emotion</span><span class="text-[11px] font-black text-white">${data.dominant_emotion}</span></div>
                </div>
                <h4 class="text-[9px] font-bold mt-1.5 mb-1 text-blue-300 uppercase tracking-widest px-1 shrink-0">AI Coaching Verdict</h4>
            `;

            if (data.coaching && data.coaching.length > 0) {
                data.coaching.forEach(c => {
                    insightsContainer.innerHTML += `<div class="bg-white/10 rounded-md px-2 py-1.5 flex items-start gap-1.5 mb-1 shrink-0"><span class="text-[9px] mt-0.5 opacity-70">👉</span><p class="text-[10px] text-blue-50 font-medium leading-tight">${c}</p></div>`;
                });
            }

            if (data.processed_video_url) {
                videoPreview.src = data.processed_video_url + '?t=' + new Date().getTime();
                videoPreview.controls = true;
                videoPreview.muted = false;
                videoPreview.play();
            }

            renderEnergyBars(data.energy_contour);

        } catch (err) {
            clearInterval(intv);
            setLoadState('Error occurred', 0);
            alert('Analysis Error: ' + err.message);
        }
    }
  </script>
</body>
</html>
"""

with open("templates/index.html", "w") as f:
    f.write(html_content)
print("Saved premium HTML template.")


Saved premium HTML template.


In [3]:
import librosa
import cv2
import numpy as np
import tensorflow as tf
from huggingface_hub import hf_hub_download
import moviepy.editor as mp
import whisper
import os
import urllib.request
import string
import cmudict
import Levenshtein
import torch
from collections import Counter
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from flask import Flask, render_template, request, jsonify
from werkzeug.serving import make_server
import threading
from google.colab.output import eval_js

print("Downloading models from Hugging Face Hub...")

cnn_model_path = hf_hub_download(repo_id="iamnotpalak/personapath-mobilenetv2-emotion-fer2013", filename="cnn_emotion_phase4.keras")
lstm_model_path = hf_hub_download(repo_id="iamnotpalak/personapath-lstm-vocal-confidence-mfcc", filename="lstm_confidence_final.keras")
scaler_mean_path = hf_hub_download(repo_id="iamnotpalak/personapath-lstm-vocal-confidence-mfcc", filename="scaler_mean.npy")
scaler_scale_path = hf_hub_download(repo_id="iamnotpalak/personapath-lstm-vocal-confidence-mfcc", filename="scaler_scale.npy")

print("Loading AI Pipelines (Wav2Vec2 + LSTM + CNN + Whisper)...")
# Phase 1 Models
device_w2v = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
processor_w2v = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')
model_w2v = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-base-960h')
model_w2v.to(device_w2v)
cmu = cmudict.dict()

# Phase 2 Models
lstm_model = tf.keras.models.load_model(lstm_model_path)
scaler_mean = np.load(scaler_mean_path)
scaler_scale = np.load(scaler_scale_path)

def build_emotion_model():
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.Resizing(224, 224),
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.1),
    ])
    base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=(None, None, 3)),
        data_augmentation,
        tf.keras.layers.Lambda(lambda x: tf.keras.applications.mobilenet_v2.preprocess_input(x)),
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(7, activation='softmax')
    ])
    return model

cnn_model = build_emotion_model()
cnn_model.load_weights(cnn_model_path)
whisper_model = whisper.load_model("tiny")

if not os.path.exists("haarcascade_frontalface_default.xml"):
    cascade_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
    urllib.request.urlretrieve(cascade_url, "haarcascade_frontalface_default.xml")
face_cascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")
EMOTIONS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']


# ==========================================
# PHASE 1 CORE NLP FUNCTIONS
# ==========================================
def clean_word(word):
    return word.lower().translate(str.maketrans('', '', string.punctuation))

def get_phonemes(word):
    w = clean_word(word)
    return [p.rstrip('012') for p in cmu[w][0]] if w in cmu else None

def phoneme_distance(w1, w2):
    p1, p2 = get_phonemes(w1), get_phonemes(w2)
    if p1 is None or p2 is None:
        return None
    return Levenshtein.distance(' '.join(p1), ' '.join(p2))

def get_score(ref_w, spoken_w):
    if ref_w == spoken_w:
        return 'green'
    dist = phoneme_distance(ref_w, spoken_w)
    if dist is None:
        dist = Levenshtein.distance(ref_w, spoken_w)
    similarity = Levenshtein.ratio(ref_w, spoken_w)
    if similarity > 0.55:
        return 'grey'
    if dist is not None and dist <= 4:
        return 'grey'
    return 'red'

def normalize_word(word):
    for sfx, trim in [('ing', 3), ('ed', 2), ('ly', 2)]:
        if word.endswith(sfx):
            return word[:-trim]
    return word

def generate_tip(word_scores):
    red_words = [ws['word'] for ws in word_scores if ws['score'] == 'red']
    grey_words = [ws['word'] for ws in word_scores if ws['score'] == 'grey']
    if not red_words and not grey_words:
        return 'Perfect reading! Every word was spot-on. Amazing work!'
    if red_words:
        focus = red_words[0]
        return f"Great job! Let's try saying '{focus}' again slowly. Break it into syllables. You're doing amazing!"
    focus = grey_words[0]
    return f"Very nice! Try pronouncing '{focus}' one more time - almost perfect! Keep going!"

def pronunciation_pipeline(audio_path, reference_text):
    speech, sr = librosa.load(audio_path, sr=16000)
    duration_sec = librosa.get_duration(y=speech, sr=sr)
    duration_min = duration_sec / 60

    inputs = processor_w2v(speech, return_tensors='pt', sampling_rate=16000).input_values.to(device_w2v)
    with torch.no_grad():
        logits = model_w2v(inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor_w2v.decode(predicted_ids[0]).lower()
    spoken_words = transcription.split()
    ref_words = [clean_word(w) for w in reference_text.split()]

    editops = Levenshtein.editops(ref_words, spoken_words)
    word_scores = [{'word': w, 'spoken': w, 'score': 'green'} for w in ref_words]

    def safe_get(lst, idx):
        return lst[idx] if idx < len(lst) else ''

    for op, i, j in editops:
        if op == 'replace':
            ref_w, spoken_w = ref_words[i], safe_get(spoken_words, j)
            if not spoken_w:
                word_scores[i].update(score='red', spoken='')
                continue
            ref_n, sp_n = normalize_word(ref_w), normalize_word(spoken_w)
            if Levenshtein.ratio(ref_n, sp_n) < 0.2:
                word_scores[i].update(score='red', spoken='')
                continue
            word_scores[i].update(score=get_score(ref_n, sp_n), spoken=spoken_w)
        elif op == 'delete':
            ref_w = ref_words[i]
            color = 'grey' if ref_w in {'the','a','and','to','of','in','on'} else 'red'
            word_scores[i].update(score=color, spoken='')

    wpm = len(spoken_words) / duration_min if duration_min > 0 else 0
    intervals = librosa.effects.split(speech, top_db=25)
    hesitations = sum(
        1 for k in range(1, len(intervals))
        if (intervals[k][0] - intervals[k-1][1]) / sr > 0.5
    )
    tip = generate_tip(word_scores)
    green = sum(1 for ws in word_scores if ws['score'] == 'green')
    total = len(word_scores)
    accuracy = round((green / total) * 100) if total else 0

    return {
        'transcription': transcription,
        'word_scores': word_scores,
        'wpm': round(wpm, 1),
        'hesitations': hesitations,
        'tip': tip,
        'accuracy': accuracy,
        'duration': round(duration_sec, 1),
    }


# ==========================================
# PHASE 2 CORE FUNCTIONS
# ==========================================
def extract_features(y, sr, n_mfcc=13, max_frames=200):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    pitch = librosa.yin(y, fmin=50, fmax=500)[np.newaxis, :]
    pitch = np.nan_to_num(pitch, nan=0.0)
    rms = librosa.feature.rms(y=y)
    zcr = librosa.feature.zero_crossing_rate(y)
    t = mfcc.shape[1]
    features = np.vstack([mfcc, delta, delta2, pitch[:, :t], rms[:, :t], zcr[:, :t]]).T
    if features.shape[0] >= max_frames: features = features[:max_frames]
    else: features = np.vstack([features, np.zeros((max_frames - features.shape[0], features.shape[1]))])
    return features.astype(np.float32)

def analyze_presentation(video_path):
    cap = cv2.VideoCapture(video_path)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter('temp_video.mp4', fourcc, fps, (width, height))

    emotions_counter = Counter()
    while True:
        ret, frame = cap.read()
        if not ret: break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.2, 5, minSize=(60, 60))
        for (x, y, w, h) in faces:
            face_img = frame[y:y+h, x:x+w]
            preds = cnn_model.predict(np.expand_dims(cv2.resize(cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB), (224, 224)).astype(np.float32), axis=0), verbose=0)[0]
            emotion = EMOTIONS[np.argmax(preds)]
            emotions_counter[emotion] += 1

            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
            cv2.putText(frame, emotion, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
            break
        out.write(frame)
    cap.release()
    out.release()

    audio_path = "temp_audio.wav"
    try:
        video = mp.VideoFileClip(video_path)
        if video.audio is None: return {"error": "Video has no audio."}
        video.audio.write_audiofile(audio_path, logger=None)
    except Exception as e:
        return {"error": f"Audio extraction failed: {str(e)}"}

    os.makedirs("static", exist_ok=True)
    os.system(f"ffmpeg -y -i temp_video.mp4 -i {audio_path} -c:v libx264 -c:a aac -strict experimental static/processed_video.mp4")

    y, sr = librosa.load(audio_path, sr=16000)
    duration = librosa.get_duration(y=y, sr=sr)
    intervals = librosa.effects.split(y, top_db=30)
    voiced_secs = sum((end - start) for start, end in intervals) / sr
    silence_ratio = 1 - (voiced_secs / duration) if duration > 0 else 1
    rms_arr = librosa.feature.rms(y=y)[0]
    rms_mean = float(np.mean(rms_arr))

    energy_contour = []
    if len(rms_arr) >= 20:
        bins = np.array_split(rms_arr, 20)
        energy_contour = [float(np.mean(b)) for b in bins]
    else:
        energy_contour = [float(x) for x in rms_arr] + [0.0]*(20-len(rms_arr))

    max_e = max(energy_contour) if max(energy_contour) > 0 else 1
    energy_contour = [round(e/max_e, 2) for e in energy_contour]

    f0 = librosa.yin(y, fmin=50, fmax=500)
    pitch_std = float(np.std(f0[f0 > 0])) if len(f0[f0 > 0]) > 10 else 0.0

    result = whisper_model.transcribe(audio_path, initial_prompt="Um, uh, like, so, you know.")
    words = [w.strip('.,?!;') for w in result['text'].lower().split()]
    filler_counts = {fw: words.count(fw) for fw in ['um', 'uh', 'like', 'so']}
    wpm = int((len(words) / (result['segments'][-1]['end'] if result['segments'] else 1.0)) * 60) if words else 0

    features = np.expand_dims(((extract_features(y, sr) - scaler_mean) / scaler_scale).astype(np.float32), axis=0)
    raw_score = float(lstm_model.predict(features, verbose=0)[0][0]) * 100

    adjusted_score = raw_score
    if silence_ratio > 0.35: adjusted_score -= 10
    if rms_mean < 0.02: adjusted_score -= 8
    if pitch_std < 20: adjusted_score -= 7
    total_fillers = sum(filler_counts.values())
    if total_fillers > 5: adjusted_score -= (total_fillers - 5) * 1.5

    adjusted_score = round(max(0, min(100, adjusted_score)), 1)

    coaching = []
    if adjusted_score > 80: coaching.append("Excellent overall delivery and confidence.")
    elif adjusted_score > 60: coaching.append("Good delivery, but room for improvement in pacing and vocal energy.")
    else: coaching.append("Delivery appears hesitant. Focus on steady pacing and reducing filler words.")

    if wpm < 110: coaching.append(f"Pace is a bit slow ({wpm} wpm). Try to reach ~130 wpm.")
    elif wpm > 160: coaching.append(f"Pace is very fast ({wpm} wpm). Slow down after key points.")

    if total_fillers > 2: coaching.append(f"Try to reduce filler words. You used {total_fillers} in this short clip.")

    if pitch_std < 15: coaching.append("Vocal tone is somewhat monotone. Try varying your pitch to sound more engaging.")

    return {
        "raw_confidence": round(raw_score, 1),
        "adjusted_confidence": adjusted_score,
        "wpm": wpm,
        "silence_ratio": round(silence_ratio, 2),
        "pitch_std": round(pitch_std, 1),
        "dominant_emotion": emotions_counter.most_common(1)[0][0] if emotions_counter else "Neutral",
        "filler_counts": filler_counts,
        "energy_contour": energy_contour,
        "coaching": coaching,
        "processed_video_url": "/static/processed_video.mp4"
    }


# ==========================================
# FLASK WEB APP DEFINITIONS
# ==========================================
app = Flask(__name__, static_folder='static', static_url_path='/static')

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/analyze_phase1', methods=['POST'])
def analyze_phase1_route():
    file = request.files.get('audio')
    ref_text = request.form.get('reference_text', '').strip()
    if not file:
        return jsonify({"error": "No audio file uploaded"}), 400
    if not ref_text:
        return jsonify({"error": "No reference text provided"}), 400

    file_path = "uploaded_audio.wav"
    file.save(file_path)

    try:
        result = pronunciation_pipeline(file_path, ref_text)
        return jsonify(result)
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/analyze', methods=['POST'])
def analyze_route():
    file = request.files.get('video')
    if not file: return jsonify({"error": "No video uploaded"}), 400
    file_path = "uploaded_video.mp4"
    file.save(file_path)
    return jsonify(analyze_presentation(file_path))

import socket

def get_free_port():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.bind(('127.0.0.1', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = get_free_port()

class ServerThread(threading.Thread):
    def __init__(self, app, port):
        threading.Thread.__init__(self)
        self.server = make_server('127.0.0.1', port, app)
        self.ctx = app.app_context()
        self.ctx.push()
    def run(self): self.server.serve_forever()
    def shutdown(self): self.server.shutdown()

server = ServerThread(app, port)
server.start()

print("\n" + "="*60)
print("FULL-SCREEN DASHBOARD IS RUNNING")
print("Click the link below to open the Web App:")
print(eval_js(f"google.colab.kernel.proxyPort({port})"))
print("="*60 + "\n")


Loading AI Pipeline (LSTM + CNN + Whisper)...


  saveable.load_own_variables(weights_store.get(inner_path))

  warnings.warn(




FULL-SCREEN DASHBOARD IS RUNNING
Click the link below to open the Web App:
https://46609-gpu-t4-s-kkb-usw4a2-q4xhth81tax3-a.us-west4-2.prod.colab.dev

